In [1]:
from pathlib import Path

import networkx as nx

import onnx
import onnxsim

from onnxnet.process.onnx_graph_utils import onnx_to_graph

In [2]:
file = Path("data/hnasbench201_simplify/cifar10/seed=0_0.onnx")
onnx_model = onnx.load(file)
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx_model = onnxsim.simplify(onnx_model)[0]
graph = onnx_to_graph(onnx_model)
graph

/home/alex/Documents/Projects/ONNX-Net/src/onnxnet/process/onnx_graph_utils.py:643: UserWarning: Multiple input nodes found.
  warnings.warn(msg, stacklevel=1)


digraph {
    node0[label="Input()  # (1,16,32,32)"]
    node1[label="Parameter((16))"]
    node2[label="Parameter((16))"]
    node3[label="Parameter((32))"]
    node4[label="Parameter((32))"]
    node5[label="Parameter((1,16,32,32))"]
    node6[label="Parameter((1,32,16,16))"]
    node7[label="Conv2D(inc=16,outc=16,size=3)"]
    node8[label="AvgPool2D(size=3)"]
    node9[label="Conv2D(inc=16,outc=16,size=1)"]
    node10[label="Add"]
    node11[label="Conv2D(inc=16,outc=16,size=1)"]
    node12[label="AvgPool2D(size=3)"]
    node13[label="Add"]
    node14[label="Add"]
    node15[label="AvgPool2D(size=3)"]
    node16[label="Relu"]
    node17[label="Conv2D(inc=16,outc=16,size=3)"]
    node18[label="Conv2D(inc=16,outc=16,size=3)"]
    node19[label="LayerNorm"]
    node20[label="Add"]
    node21[label="AvgPool2D(size=3)"]
    node22[label="Add"]
    node23[label="Add"]
    node24[label="Add"]
    node25[label="Relu"]
    node26[label="Conv2D(inc=16,outc=16,size=3)"]
    node27[label="Instan

In [ ]:
import numpy as np
import scipy as sp
import torch
from torch import nn
import torch.nn.functional as F

with np.printoptions(linewidth=2000, threshold=100000):
    A1 = graph.adjacency_matrix()
    A2 = nx.to_scipy_sparse_array(graph.to_networkx())
    D1 = graph.degree_matrix(direction="out", insert_self_loops=True)
    D2 = np.diag([x[1] for x in graph.to_networkx().out_degree(weight='weight')])
    # print(np.all(A1 == A2))
    # print(np.where(A1 != A2))
    # print(A1)
    # print(A2)
    # print(np.all(D1 == D2))
    # print(np.where(D1 != D2))
    # print(D1)
    # print(D2)

random_projection = torch.randn(512, 256) / np.sqrt(256)

adjacency = graph.adjacency_matrix()
out_degrees = graph.degree_matrix(direction="out", insert_self_loops=True)
in_degrees = graph.degree_matrix(direction="in", insert_self_loops=True)
# Use "SVDFormer" structure matrix with "Edge-augmented Graph Transformer"'s SVD approach
matrix = out_degrees.power(-0.5) @ (adjacency + sp.sparse.identity(len(graph.nodes))) @ in_degrees.power(-0.5)

U, S, Vt = sp.sparse.linalg.svds(matrix, k=100, which="LM")
idx = np.argsort(S)[::-1]
U, S, Vt = U[:, idx], S[idx], Vt[idx]  # pyright: ignore [reportConstantRedefinition]
V = Vt.T
signs = np.sign(U.sum(axis=0))
signs[signs == 0] = 1
U *= signs  # pyright: ignore [reportConstantRedefinition]
V *= signs  # pyright: ignore [reportConstantRedefinition]
U /= np.linalg.norm(U, axis=0, keepdims=True)  # pyright: ignore [reportConstantRedefinition]
V /= np.linalg.norm(V, axis=0, keepdims=True)  # pyright: ignore [reportConstantRedefinition]
enc = torch.from_numpy(np.concat([U, V], axis=1)).to(torch.get_default_dtype())
enc = F.pad(enc[: ,:256], (0, max(0, 256 - enc.shape[1])))
enc = random_projection(enc)
print(enc.shape)
enc

torch.Size([153, 512])


tensor([[ 0.0060,  0.0065, -0.0507,  ..., -0.0284,  0.0084,  0.0370],
        [-0.0593,  0.0318, -0.0412,  ..., -0.0322,  0.0487,  0.0169],
        [-0.0593,  0.0318, -0.0412,  ..., -0.0322,  0.0487,  0.0169],
        ...,
        [-0.0770,  0.1304, -0.0372,  ..., -0.0387, -0.0416,  0.0578],
        [-0.0631,  0.0631, -0.0617,  ..., -0.0368,  0.0544,  0.0055],
        [-0.0256, -0.0054, -0.0629,  ..., -0.0599,  0.1197,  0.0086]],
       grad_fn=<AddmmBackward0>)